# Database provisioning cloud-run analysis

This notebook queries uploaded AX experiment data with the `ax query` CLI and visualizes completion, agent runtime, and tool-call steps for the Codex + GPT-5.6-luna cloud sample.

- Experiment: `database-provisioning`
- Run request: `01KZPMF9YMW933F9X4E9KMEPA3`
- Sample: 10 runs each for MongoDB, PostgreSQL, and SQLite
- Date: 2026-08-10

The notebook deliberately queries derived metadata and boolean evidence rather than raw transcript text, which may contain generated database credentials.

In [ ]:
# Prerequisite (run once if needed):
# %pip install pandas matplotlib

In [ ]:
import json
import shlex
import shutil
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RUN_REQUEST_ID = "01KZPMF9YMW933F9X4E9KMEPA3"
DATABASE_ORDER = ["mongodb", "postgresql", "sqlite"]
DISPLAY_NAMES = {
    "mongodb": "MongoDB",
    "postgresql": "PostgreSQL",
    "sqlite": "SQLite",
}

assert shutil.which("ax"), "Install the AX CLI and make sure `ax` is on PATH."


def ax_query(sql: str, *, limit: int = 10_000) -> pd.DataFrame:
    """Run an authenticated AX Cloud query and parse its NDJSON output."""
    command = [
        "ax",
        "query",
        sql,
        "--format",
        "json",
        "--limit",
        str(limit),
    ]
    print("$", shlex.join(command))
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    rows = [json.loads(line) for line in result.stdout.splitlines() if line.strip()]
    return pd.DataFrame(rows)


plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## Query per-run metrics

The next cell executes the equivalent of:

```bash
ax query "<ClickHouse SQL>" --format json --limit 10000
```

It filters by the exact cloud run request so uploaded local smoke runs and earlier cloud runs cannot affect the results. `tool_calls_total` is used as the comparable step proxy; AX `num_turns` counts low-level harness events rather than conversational steps.

In [ ]:
per_run_sql = f"""
SELECT
    run_id,
    prompt_id AS database,
    repeat_idx,
    status,
    exit_reason,
    dateDiff(
        'millisecond',
        parseDateTime64BestEffort(agent_started_at),
        parseDateTime64BestEffort(agent_ended_at)
    ) / 1000.0 AS runtime_s,
    tool_calls_total,
    tool_calls_succeeded,
    tool_calls_failed,
    input_tokens,
    output_tokens,
    cost_usd_micros / 1000000.0 AS cost_usd
FROM runs
WHERE run_request_id = '{RUN_REQUEST_ID}'
ORDER BY database, repeat_idx
"""

runs = ax_query(per_run_sql)
assert len(runs) == 30, f"Expected 30 runs, received {len(runs)}"
assert set(runs["database"]) == set(DATABASE_ORDER)

numeric_columns = [
    "repeat_idx",
    "runtime_s",
    "tool_calls_total",
    "tool_calls_succeeded",
    "tool_calls_failed",
    "input_tokens",
    "output_tokens",
    "cost_usd",
]
runs[numeric_columns] = runs[numeric_columns].apply(pd.to_numeric)
runs["database"] = pd.Categorical(
    runs["database"], categories=DATABASE_ORDER, ordered=True
)
runs.head()

## Classify final-response completion

The in-sandbox `uri-reported` test is currently blocked because `ax-run-query` is unavailable. This post-hoc query reads only the final `agent_message` for each run and returns a boolean indicating whether it contains a database-appropriate URI. It does not return the URI itself, avoiding accidental credential exposure.

In [ ]:
completion_sql = f"""
WITH finals AS (
    SELECT
        run_id,
        argMax(coalesce(text, ''), seq_start) AS final_text
    FROM agent_timeline_entries
    WHERE entry_kind = 'agent_message'
    GROUP BY run_id
)
SELECT
    r.run_id,
    toUInt8(multiIf(
        r.prompt_id = 'mongodb',
            positionCaseInsensitive(f.final_text, 'mongodb://') > 0
            OR positionCaseInsensitive(f.final_text, 'mongodb+srv://') > 0,
        r.prompt_id = 'postgresql',
            positionCaseInsensitive(f.final_text, 'postgres://') > 0
            OR positionCaseInsensitive(f.final_text, 'postgresql://') > 0,
        r.prompt_id = 'sqlite',
            positionCaseInsensitive(f.final_text, 'sqlite:') > 0
            OR positionCaseInsensitive(f.final_text, 'file:/workspace/') > 0
            OR positionCaseInsensitive(f.final_text, '/workspace/') > 0,
        false
    )) AS uri_reported
FROM runs r
INNER JOIN finals f ON r.run_id = f.run_id
WHERE r.run_request_id = '{RUN_REQUEST_ID}'
"""

completion = ax_query(completion_sql)
completion["uri_reported"] = pd.to_numeric(completion["uri_reported"]).astype(bool)
runs = runs.merge(completion, on="run_id", how="inner", validate="one_to_one")

runs.groupby("database", observed=True)["uri_reported"].agg(["sum", "count", "mean"])

In [ ]:
summary = (
    runs.groupby("database", observed=True)
    .agg(
        runs=("run_id", "size"),
        uri_completion=("uri_reported", "mean"),
        runtime_mean_s=("runtime_s", "mean"),
        runtime_sd_s=("runtime_s", "std"),
        runtime_p50_s=("runtime_s", "median"),
        runtime_p95_s=("runtime_s", lambda values: values.quantile(0.95)),
        tool_calls_mean=("tool_calls_total", "mean"),
        tool_calls_sd=("tool_calls_total", "std"),
        tool_calls_min=("tool_calls_total", "min"),
        tool_calls_max=("tool_calls_total", "max"),
        cost_total_usd=("cost_usd", "sum"),
    )
    .reset_index()
)

# Two-sided 95% t interval for n=10 (df=9).
t_critical_df9 = 2.262157
summary["runtime_mean_ci95_s"] = (
    t_critical_df9 * summary["runtime_sd_s"] / np.sqrt(summary["runs"])
)
summary["tool_calls_mean_ci95"] = (
    t_critical_df9 * summary["tool_calls_sd"] / np.sqrt(summary["runs"])
)
summary["uri_completion_pct"] = summary["uri_completion"] * 100
summary["database_label"] = summary["database"].map(DISPLAY_NAMES)

summary[
    [
        "database_label",
        "runs",
        "uri_completion_pct",
        "runtime_mean_s",
        "runtime_sd_s",
        "runtime_p50_s",
        "runtime_p95_s",
        "tool_calls_mean",
        "tool_calls_sd",
        "tool_calls_min",
        "tool_calls_max",
        "cost_total_usd",
    ]
].round(2)

## Visualize completion, runtime, and steps

Error bars on mean runtime and mean tool calls are 95% t-intervals. The scatter plot shows every run so variance and outliers remain visible rather than being hidden by averages.

In [ ]:
plot_summary = summary.set_index("database").loc[DATABASE_ORDER]
labels = [DISPLAY_NAMES[name] for name in DATABASE_ORDER]
colors = {
    "mongodb": "#00A35C",
    "postgresql": "#336791",
    "sqlite": "#8C8C8C",
}
bar_colors = [colors[name] for name in DATABASE_ORDER]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
fig.suptitle(
    "Database provisioning — Codex + GPT-5.6-luna (10 cloud runs per DB)",
    fontsize=16,
    fontweight="bold",
)

axes[0, 0].bar(
    labels,
    plot_summary["runtime_mean_s"],
    yerr=plot_summary["runtime_mean_ci95_s"],
    capsize=5,
    color=bar_colors,
)
axes[0, 0].set_title("Mean agent runtime with 95% CI")
axes[0, 0].set_xlabel("Database")
axes[0, 0].set_ylabel("Agent runtime (seconds)")

axes[0, 1].bar(
    labels,
    plot_summary["tool_calls_mean"],
    yerr=plot_summary["tool_calls_mean_ci95"],
    capsize=5,
    color=bar_colors,
)
axes[0, 1].set_title("Mean tool-call steps with 95% CI")
axes[0, 1].set_xlabel("Database")
axes[0, 1].set_ylabel("Tool calls per run")

axes[1, 0].bar(
    labels,
    plot_summary["uri_completion_pct"],
    color=bar_colors,
)
axes[1, 0].set_title("Final-response URI completion")
axes[1, 0].set_xlabel("Database")
axes[1, 0].set_ylabel("Runs reporting a URI (%)")
axes[1, 0].set_ylim(0, 105)
for index, value in enumerate(plot_summary["uri_completion_pct"]):
    axes[1, 0].text(index, value - 6, f"{value:.0f}%", ha="center", color="white", fontweight="bold")

for database in DATABASE_ORDER:
    subset = runs[runs["database"] == database]
    axes[1, 1].scatter(
        subset["tool_calls_total"],
        subset["runtime_s"],
        label=DISPLAY_NAMES[database],
        color=colors[database],
        s=65,
        alpha=0.8,
    )
axes[1, 1].set_title("Individual-run runtime versus tool calls")
axes[1, 1].set_xlabel("Tool calls (count)")
axes[1, 1].set_ylabel("Agent runtime (seconds)")
axes[1, 1].legend(title="Database")

plt.show()

## MongoDB transcript evidence audit

A reported URI is not proof that a database was actually running. This query classifies sanitized command evidence for each MongoDB run:

- **Protocol ping:** a completed tool invoked a MongoDB `ping` and returned output containing `ok`.
- **TCP probe:** a completed `/dev/tcp` connection to port 27017 succeeded.
- **Listener/process:** completed command output showed a listener on port 27017 and a `mongod` process.

The query returns booleans only; raw command input/output is intentionally excluded because it can contain generated credentials.

In [ ]:
mongodb_evidence_sql = f"""
SELECT
    r.repeat_idx,
    toUInt8(countIf(
        t.status = 'completed'
        AND positionCaseInsensitive(coalesce(t.raw_input, ''), 'ping') > 0
        AND positionCaseInsensitive(coalesce(t.raw_output, ''), 'ok') > 0
        AND positionCaseInsensitive(coalesce(t.raw_output, ''), 'not bundled') = 0
    ) > 0) AS protocol_ping,
    toUInt8(countIf(
        t.status = 'completed'
        AND positionCaseInsensitive(coalesce(t.raw_input, ''), '/dev/tcp') > 0
        AND (
            positionCaseInsensitive(coalesce(t.raw_output, ''), 'tcp_connected') > 0
            OR positionCaseInsensitive(coalesce(t.raw_output, ''), 'tcp-connect: ok') > 0
        )
    ) > 0) AS tcp_probe,
    toUInt8(countIf(
        t.status = 'completed'
        AND positionCaseInsensitive(coalesce(t.raw_output, ''), 'listen') > 0
        AND positionCaseInsensitive(coalesce(t.raw_output, ''), '27017') > 0
    ) > 0) AS listener_evidence,
    toUInt8(countIf(
        t.status = 'completed'
        AND positionCaseInsensitive(coalesce(t.raw_output, ''), 'mongod') > 0
    ) > 0) AS process_evidence
FROM tool_calls t
INNER JOIN runs r ON t.run_id = r.run_id
WHERE r.run_request_id = '{RUN_REQUEST_ID}'
  AND r.prompt_id = 'mongodb'
GROUP BY r.repeat_idx
ORDER BY r.repeat_idx
"""

mongodb_evidence = ax_query(mongodb_evidence_sql)
evidence_columns = [
    "repeat_idx",
    "protocol_ping",
    "tcp_probe",
    "listener_evidence",
    "process_evidence",
]
mongodb_evidence[evidence_columns] = mongodb_evidence[evidence_columns].apply(pd.to_numeric)
mongodb_evidence["evidence_tier"] = np.select(
    [
        mongodb_evidence["protocol_ping"].eq(1),
        mongodb_evidence["tcp_probe"].eq(1)
        & mongodb_evidence["listener_evidence"].eq(1)
        & mongodb_evidence["process_evidence"].eq(1),
    ],
    ["MongoDB protocol verified", "TCP/listener verified"],
    default="Insufficient transcript evidence",
)

mongodb_evidence

In [ ]:
tier_order = [
    "MongoDB protocol verified",
    "TCP/listener verified",
    "Insufficient transcript evidence",
]
tier_counts = mongodb_evidence["evidence_tier"].value_counts().reindex(tier_order, fill_value=0)

fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
bars = ax.barh(tier_counts.index, tier_counts.values, color=["#00A35C", "#E3A008", "#C73E1D"])
ax.set_title("MongoDB provisioning evidence from 10 cloud transcripts")
ax.set_xlabel("Runs (count)")
ax.set_ylabel("Strongest observed evidence tier")
ax.set_xlim(0, 10)
ax.bar_label(bars, padding=4)
plt.show()

protocol_verified = int(tier_counts["MongoDB protocol verified"])
tcp_verified = int(tier_counts["TCP/listener verified"])
insufficient = int(tier_counts["Insufficient transcript evidence"])
print(
    f"MongoDB audit: {protocol_verified}/10 runs had application-level ping evidence; "
    f"{tcp_verified}/10 had TCP/listener/process evidence only; "
    f"{insufficient}/10 lacked sufficient transcript evidence."
)

## MongoDB provisioning method

Provisioning method is classified from **successful executed commands**, not mere mentions or availability checks. The primary method records how `mongod` was obtained:

- **OS package manager:** a successful `apt`, `dnf`, or `yum` installation step.
- **Direct MongoDB archive:** a successful server download from MongoDB followed by direct binary use.
- **Docker/Podman:** a successful container-run command.

A run may subsequently register a directly downloaded binary as a system service; its primary acquisition method remains “Direct MongoDB archive.”

In [ ]:
mongodb_method_sql = f"""
SELECT
    r.run_id,
    r.repeat_idx,
    toUInt8(countIf(
        t.status = 'completed'
        AND (
            positionCaseInsensitive(coalesce(t.raw_input, ''), 'docker run') > 0
            OR positionCaseInsensitive(coalesce(t.raw_input, ''), 'podman run') > 0
        )
    ) > 0) AS used_container,
    toUInt8(countIf(
        t.status = 'completed'
        AND (
            positionCaseInsensitive(coalesce(t.raw_input, ''), 'apt-get install') > 0
            OR positionCaseInsensitive(coalesce(t.raw_input, ''), 'apt install') > 0
            OR positionCaseInsensitive(coalesce(t.raw_input, ''), 'dnf install') > 0
            OR positionCaseInsensitive(coalesce(t.raw_input, ''), 'yum install') > 0
        )
    ) > 0) AS used_package_manager,
    toUInt8(countIf(
        t.status = 'completed'
        AND (
            positionCaseInsensitive(coalesce(t.raw_input, ''), 'fastdl.mongodb.org') > 0
            OR positionCaseInsensitive(coalesce(t.raw_input, ''), 'downloads.mongodb.org') > 0
        )
        AND positionCaseInsensitive(coalesce(t.raw_input, ''), 'mongod') > 0
    ) > 0) AS used_server_archive
FROM tool_calls t
INNER JOIN runs r ON t.run_id = r.run_id
WHERE r.run_request_id = '{RUN_REQUEST_ID}'
  AND r.prompt_id = 'mongodb'
GROUP BY r.run_id, r.repeat_idx
ORDER BY r.repeat_idx
"""

mongodb_methods = ax_query(mongodb_method_sql)
method_flags = ["used_container", "used_package_manager", "used_server_archive"]
mongodb_methods[["repeat_idx", *method_flags]] = mongodb_methods[
    ["repeat_idx", *method_flags]
].apply(pd.to_numeric)

mongodb_methods["provisioning_method"] = np.select(
    [
        mongodb_methods["used_container"].eq(1),
        mongodb_methods["used_package_manager"].eq(1),
        mongodb_methods["used_server_archive"].eq(1),
    ],
    ["Docker/Podman", "OS package manager", "Direct MongoDB archive"],
    default="Unknown",
)

method_order = [
    "Direct MongoDB archive",
    "OS package manager",
    "Docker/Podman",
    "Unknown",
]
method_counts = (
    mongodb_methods["provisioning_method"]
    .value_counts()
    .reindex(method_order, fill_value=0)
)
method_counts

In [ ]:
mongodb_method_runs = mongodb_methods.merge(
    runs.loc[
        runs["database"] == "mongodb",
        ["repeat_idx", "runtime_s", "cost_usd", "tool_calls_total"],
    ],
    on="repeat_idx",
    how="inner",
    validate="one_to_one",
)
observed_methods = [method for method in method_order if method_counts[method] > 0]
method_means = (
    mongodb_method_runs.groupby("provisioning_method")
    .agg(
        runtime_s=("runtime_s", "mean"),
        cost_usd=("cost_usd", "mean"),
        tool_calls_total=("tool_calls_total", "mean"),
    )
    .reindex(observed_methods)
)
method_colors = {
    "Direct MongoDB archive": "#00A35C",
    "OS package manager": "#336791",
    "Docker/Podman": "#8C8C8C",
    "Unknown": "#C73E1D",
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
fig.suptitle("MongoDB provisioning method from executed commands", fontsize=15, fontweight="bold")

count_bars = axes[0, 0].barh(
    method_counts.index,
    method_counts.values,
    color=[method_colors[method] for method in method_counts.index],
)
axes[0, 0].set_title("Primary provisioning method")
axes[0, 0].set_xlabel("Runs (count)")
axes[0, 0].set_ylabel("Method")
axes[0, 0].set_xlim(0, 10)
axes[0, 0].bar_label(count_bars, padding=4)

rng = np.random.default_rng(514)


def plot_method_metric(ax, column, title, ylabel, label_format):
    means = method_means[column]
    bars = ax.bar(
        observed_methods,
        means.values,
        color=[method_colors[method] for method in observed_methods],
        alpha=0.75,
    )
    for index, method in enumerate(observed_methods):
        values = mongodb_method_runs.loc[
            mongodb_method_runs["provisioning_method"] == method, column
        ]
        jitter = rng.uniform(-0.08, 0.08, size=len(values))
        ax.scatter(
            np.full(len(values), index) + jitter,
            values,
            color="black",
            alpha=0.7,
            s=35,
            zorder=3,
        )
    ax.bar_label(
        bars,
        labels=[label_format(value) for value in means],
        padding=4,
    )
    ax.set_title(title)
    ax.set_xlabel("Primary provisioning method")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=12)


plot_method_metric(
    axes[0, 1],
    "runtime_s",
    "Runtime by observed method",
    "Agent runtime (seconds)",
    lambda value: f"{value:.1f}s",
)
plot_method_metric(
    axes[1, 0],
    "cost_usd",
    "Cost by observed method",
    "Model cost per run (USD)",
    lambda value: f"${value:.3f}",
)
plot_method_metric(
    axes[1, 1],
    "tool_calls_total",
    "Tool-call steps by observed method",
    "Tool calls per run (count)",
    lambda value: f"{value:.1f}",
)

plt.show()

method_means.rename(
    columns={
        "runtime_s": "mean_runtime_s",
        "cost_usd": "mean_cost_usd",
        "tool_calls_total": "mean_tool_calls",
    }
).round(3)

### Why Docker was not used

The transcripts show agents considering Docker, but not failing during container provisioning. They probed for Docker/Compose/Podman first, found no usable Docker CLI, and pivoted before issuing a container-start command.

The query below returns only boolean classifications. It does not return raw transcript payloads.

In [ ]:
docker_probe_sql = f"""
SELECT
    r.repeat_idx,
    toUInt8(countIf(
        positionCaseInsensitive(coalesce(t.raw_input, ''), 'command -v docker') > 0
        OR positionCaseInsensitive(coalesce(t.raw_input, ''), 'docker version') > 0
        OR positionCaseInsensitive(coalesce(t.raw_input, ''), 'docker info') > 0
        OR positionCaseInsensitive(coalesce(t.raw_input, ''), 'docker ps') > 0
        OR positionCaseInsensitive(coalesce(t.raw_input, ''), 'docker compose version') > 0
    ) > 0) AS probed_docker_cli,
    toUInt8(countIf(
        positionCaseInsensitive(coalesce(t.raw_output, ''), 'docker: command not found') > 0
    ) > 0) AS explicit_command_not_found,
    toUInt8(countIf(
        positionCaseInsensitive(coalesce(t.raw_input, ''), 'docker run') > 0
        OR positionCaseInsensitive(coalesce(t.raw_input, ''), 'docker compose up') > 0
        OR positionCaseInsensitive(coalesce(t.raw_input, ''), 'podman run') > 0
    ) > 0) AS attempted_container_start,
    toUInt8(countIf(
        t.status = 'completed'
        AND (
            positionCaseInsensitive(coalesce(t.raw_input, ''), 'docker run') > 0
            OR positionCaseInsensitive(coalesce(t.raw_input, ''), 'docker compose up') > 0
            OR positionCaseInsensitive(coalesce(t.raw_input, ''), 'podman run') > 0
        )
    ) > 0) AS successful_container_start
FROM tool_calls t
INNER JOIN runs r ON t.run_id = r.run_id
WHERE r.run_request_id = '{RUN_REQUEST_ID}'
  AND r.prompt_id = 'mongodb'
GROUP BY r.repeat_idx
ORDER BY r.repeat_idx
"""

docker_probes = ax_query(docker_probe_sql)
docker_probe_columns = [
    "repeat_idx",
    "probed_docker_cli",
    "explicit_command_not_found",
    "attempted_container_start",
    "successful_container_start",
]
docker_probes[docker_probe_columns] = docker_probes[docker_probe_columns].apply(pd.to_numeric)
docker_probes

In [ ]:
docker_funnel = pd.Series(
    {
        "Probed Docker CLI": int(docker_probes["probed_docker_cli"].sum()),
        "Explicit `command not found`": int(
            docker_probes["explicit_command_not_found"].sum()
        ),
        "Attempted container start": int(
            docker_probes["attempted_container_start"].sum()
        ),
        "Successfully started container": int(
            docker_probes["successful_container_start"].sum()
        ),
    }
)

fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
bars = ax.barh(
    docker_funnel.index,
    docker_funnel.values,
    color=["#336791", "#C73E1D", "#8C8C8C", "#00A35C"],
)
ax.invert_yaxis()
ax.set_title("Docker path observed in MongoDB transcripts")
ax.set_xlabel("MongoDB runs (count)")
ax.set_ylabel("Transcript signal")
ax.set_xlim(0, 10)
ax.bar_label(bars, padding=4)
plt.show()

docker_funnel

**Finding:** all 10 MongoDB runs probed Docker tooling. Five emitted an explicit `docker: command not found`; the other probes produced no usable Docker executable. No run issued `docker run`, `docker compose up`, or `podman run`.

Therefore, “Docker failed” would be misleading: no container launch was attempted. Docker was unavailable in the base agent environment, so agents pivoted to direct MongoDB archives or OS packages. Separate `bubblewrap is unavailable` errors seen in a few initial shell invocations came from the Codex command sandbox and were not Docker daemon or MongoDB container failures.

#### Docker probe context

The isolated Docker mentions can sound like failed container deployments. The ordered transcript context shows something narrower: agents considered containers during environment discovery, probed for the CLI, observed that it was unavailable, and selected a native fallback before attempting any container launch.

The next query reconstructs that decision sequence for two representative runs without returning tool payloads or generated credentials.

In [14]:
docker_context_sql = f"""
SELECT
    r.repeat_idx,
    e.seq_start,
    e.text AS transcript_message
FROM agent_timeline_entries e
INNER JOIN runs r ON e.run_id = r.run_id
WHERE r.run_request_id = '{RUN_REQUEST_ID}'
  AND r.prompt_id = 'mongodb'
  AND r.repeat_idx IN (2, 6)
  AND e.entry_kind = 'agent_message'
  AND e.seq_start BETWEEN 40 AND 220
ORDER BY r.repeat_idx, e.seq_start
"""

docker_context = ax_query(docker_context_sql)
docker_context[["repeat_idx", "seq_start"]] = docker_context[
    ["repeat_idx", "seq_start"]
].apply(pd.to_numeric)
with pd.option_context("display.max_colwidth", None):
    display(docker_context)

$ ax query '
SELECT
    r.repeat_idx,
    e.seq_start,
    e.text AS transcript_message
FROM agent_timeline_entries e
INNER JOIN runs r ON e.run_id = r.run_id
WHERE r.run_request_id = '"'"'01KZPMF9YMW933F9X4E9KMEPA3'"'"'
  AND r.prompt_id = '"'"'mongodb'"'"'
  AND r.repeat_idx IN (2, 6)
  AND e.entry_kind = '"'"'agent_message'"'"'
  AND e.seq_start BETWEEN 40 AND 220
ORDER BY r.repeat_idx, e.seq_start
' --format json --limit 10000


,repeat_idx,seq_start,transcript_message
0,2,50,"The workspace command cannot start because the execution sandbox is missing its `bubblewrap` runtime. I need to run the inspection and Docker provisioning with elevated execution, then I’ll verify the database health and keep it running."
1,2,105,There is no Docker CLI or project configuration in the workspace. I’m checking for another installed container/runtime or local MongoDB service so I can provision the database through the available host mechanism.
2,2,154,"The host has no MongoDB or container runtime, but it does have Debian package tools and enough disk space. I’ll use a local MongoDB server installation, bind it to the host interface for connections, and configure a persistent service with authentication enabled."
3,2,218,"I’m going to install MongoDB from the official Linux distribution archive because Debian 13’s default repositories do not provide the server package. I’ll place its data and logs under `/var/lib/mongodb` and `/var/log/mongodb`, create an authenticated admin account, and launch it as a background process that remains running."
4,6,47,"The local command sandbox is unavailable because its required launcher is missing, so I’m retrying the workspace inspection with the necessary host-level permission."
5,6,81,`rg` is not installed in this environment. I’m switching to the available shell tools and checking whether Docker itself is available before creating anything.
6,6,116,"There is no project content and no Docker/Compose runtime in `/workspace`. I’m checking for a native MongoDB server or package manager next; if neither is present, the environment cannot host a running database without installing or using an external service."
7,6,171,"The host has no MongoDB binary or container runtime, but it does have Debian’s package manager. I’m checking the available MongoDB packages; installing the server natively is the remaining local path."
8,6,217,"Debian 13’s repositories do not include the MongoDB server. I’m checking the official MongoDB distribution endpoint for a compatible binary; if reachable, I’ll install it under `/opt`, run it as a background service, bind it to accept connections, and verify with a client."


**Run 2 sequence**

1. The agent planned to inspect “an existing container or deployment setup.”
2. Its first inspection failed because the Codex command sandbox lacked `bubblewrap`. Although the agent called the elevated retry “Docker provisioning,” Docker had not yet been invoked.
3. It ran `command -v docker` and `docker version`; the tool output was `docker: command not found`.
4. It concluded: “There is no Docker CLI or project configuration in the workspace,” then checked Podman, `mongod`, the OS, and package tools.
5. It selected MongoDB’s official Linux archive after finding that Debian 13 did not provide the server package.

**Run 6 sequence**

1. The agent first searched for project Compose configuration.
2. It then explicitly said it was “checking whether Docker itself is available before creating anything.”
3. It probed Docker, Compose, and Podman; Docker returned `command not found`.
4. It concluded: “There is no project content and no Docker/Compose runtime,” checked native MongoDB and package-manager options, then selected the official MongoDB binary archive.

**Interpretation:** these are environment-discovery pivots, not failed Docker deployments. Across all 10 MongoDB runs, agents probed Docker tooling, but none issued `docker run`, `docker compose up`, or `podman run`. The `bubblewrap` failures belong to Codex’s shell isolation layer and should not be classified as Docker failures.

## Interpretation and limitations

- URI completion is a post-hoc final-response measure, not the currently failing AX test result.
- Tool calls are a step proxy; they do not measure cognitive effort or normalize for different tool granularity.
- Runtime confidence intervals describe uncertainty around these 10-run sample means. They are not proof that all pairwise database differences are statistically significant.
- MongoDB transcript evidence establishes end-of-run reachability from inside the agent sandbox. It does not establish reachability from AX’s separate test sandbox or persistence after sandbox teardown.
- A future independent `connection-accepted` probe should replace transcript-derived connectivity evidence once AX exposes the required process/network namespace.